[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/diagnostic_clean.ipynb)

# MISDA — clean controlled diagnostics

This notebook evaluates the unified controlled diagnostic suite under exact observation: `Y = Z = F(X)`. Ground truth is generated from the theoretical problem and the sampled clean objective matrix `Z`, never from the observed result or from MISDA.

The thirteen cases are presented individually so that the construction, declared structural truth, and expected MISDA behavior can be understood before each result is inspected. Cases 12 and 13 are retained as documented adversarial limits of the current method.


In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import misda
import misda.benchmarks as bench

N = 300
SEED = 123
SIGMA = 0.0


## Experimental protocol

Every case follows the same controlled pipeline:

`X → Z = F(X) → Y → MISDA`

For this notebook `sigma=0`, hence `Y=Z`. The helper below centralizes the common execution procedure, while each case remains an explicit notebook section. Stable internal problem ids are used programmatically; the public case numbers and names are presentation labels.


In [ ]:
diagnostic_results = {}

def run_case(problem_id):
    problem = bench.PROBLEM_BY_ID[problem_id]
    dataset = problem.generate(N=N, seed=SEED, sigma=SIGMA)
    truth = bench.diagnostic_truth(problem, dataset.Z)
    mis_set = misda.discover(dataset.Y, name=truth["name"], seed=SEED)
    misda.evaluate(mis_set, metrics=("linear", "pareto"))
    benchmark_result = misda.benchmark(mis_set, truth)
    print(benchmark_result.report())
    mis_set.graph_plot()
    diagnostic_results[problem.id] = {
        "dataset": dataset,
        "result_obj": mis_set,
        "benchmark_obj": benchmark_result,
        "truth": truth,
    }
    return benchmark_result


## Case 1 - Independent objectives

### Purpose
Establish the no-redundancy baseline: every objective carries its own independent degree of freedom.

### Construction
Twenty independent generating variables are mapped directly to twenty objectives, `f_i = x_i`.

### Ground truth
- Original dimension: 20
- Latent dimension: 20
- Structural dimension: 20
- Generating families: 20 singletons
- Structural units: 20 singletons

### Expected behavior
MISDA should find no positive-redundancy structure and retain all twenty objectives.


In [ ]:
case_1 = run_case("independence")


## Case 2 - Complete positive redundancy

### Purpose
Test the opposite extreme: all objectives are positively redundant measurements of one generating degree of freedom.

### Construction
A single variable `x` is copied into all twenty objectives.

### Ground truth
- Original dimension: 20
- Latent dimension: 1
- Structural dimension: 1
- Generating families: one family of 20
- Structural units: one unit of 20

### Expected behavior
MISDA should reduce the objective space to one representative.


In [ ]:
case_2 = run_case("total_redundancy")


## Case 3 - Four redundant blocks

### Purpose
Test whether MISDA separates several independent redundancy groups.

### Construction
Four independent factors generate four disjoint blocks; each factor is copied into five objectives.

### Ground truth
- Original dimension: 20
- Latent dimension: 4
- Structural dimension: 4
- Generating families: four groups of 5
- Structural units: four groups of 5

### Expected behavior
MISDA should retain one representative from each block, yielding dimension 4.


In [ ]:
case_3 = run_case("blocks_4x5")


## Case 4 - Two redundant blocks

### Purpose
Repeat the block-redundancy test with fewer, larger groups.

### Construction
Two independent factors each generate ten identical objectives.

### Ground truth
- Original dimension: 20
- Latent dimension: 2
- Structural dimension: 2
- Generating families: two groups of 10
- Structural units: two groups of 10

### Expected behavior
MISDA should retain one representative from each block, yielding dimension 2.


In [ ]:
case_4 = run_case("blocks_2x10")


## Case 5 - Mixed independent and redundant objectives

### Purpose
Test a heterogeneous objective space containing both irreducible objectives and redundant blocks.

### Construction
Ten objectives are independently generated. Two additional independent factors are each replicated five times.

### Ground truth
- Original dimension: 20
- Latent dimension: 12
- Structural dimension: 12
- Generating families: ten singletons plus two groups of 5
- Structural units: ten singletons plus two groups of 5

### Expected behavior
MISDA should retain the ten independent objectives plus one representative from each redundant block.


In [ ]:
case_5 = run_case("mixed_independent_and_blocks")


## Case 6 - Nonlinear monotonic redundancy

### Purpose
Test whether one-dimensional redundancy remains identifiable when objectives are different monotonic nonlinear transformations rather than literal copies.

### Construction
One variable `x` generates twenty monotonic linear and nonlinear transforms.

### Ground truth
- Original dimension: 20
- Latent dimension: 1
- Structural dimension: 1
- Generating families: one family of 20
- Structural units: one unit of 20

### Expected behavior
MISDA should recognize the common one-dimensional structure and select one representative.


In [ ]:
case_6 = run_case("monotonic_redundancy")


## Case 7 - Antagonistic linear groups

### Purpose
Separate latent dimensionality from structural dimensionality when two positively redundant groups are in direct conflict.

### Construction
Ten objectives are copies of `x`; ten are copies of `-x`.

### Ground truth
- Original dimension: 20
- Latent dimension: 1
- Structural dimension: 2
- Generating families: two groups of 10
- Structural units: two groups of 10

### Expected behavior
The signed dependence structure is one-dimensional, but the positive-redundancy structure requires one representative from each antagonistic group.


In [ ]:
case_7 = run_case("antagonistic_linear_groups")


## Case 8 - Trade-off with redundant families

### Purpose
Test a genuinely multiobjective trade-off in which several observable families are redundant manifestations of a lower-dimensional generating space.

### Construction
Two generating variables `(a, b)` produce cost, consumption, and performance-related objective families of sizes 7, 7, and 6.

### Ground truth
- Original dimension: 20
- Latent dimension: 2
- Structural dimension: 2
- Generating families: 7 + 7 + 6
- Structural units: not uniquely declared

### Expected behavior
MISDA should recover the two-dimensional structure without treating the three generating families as three structural dimensions.


In [ ]:
case_8 = run_case("tradeoff_redundancies")


## Case 9 - Nonlinear redundant blocks

### Purpose
Extend the block-redundancy experiment from copies to nonlinear transformations.

### Construction
Four independent factors `(u, v, w, z)` each generate five nonlinear transforms.

### Ground truth
- Original dimension: 20
- Latent dimension: 4
- Structural dimension: 4
- Generating families: four groups of 5
- Structural units: four groups of 5

### Expected behavior
MISDA should recover one representative per nonlinear block.


In [ ]:
case_9 = run_case("nonlinear_blocks_4x5")


## Case 10 - Antagonistic nonlinear groups

### Purpose
Combine nonlinear redundancy with antagonistic structure.

### Construction
Ten monotonic transforms are generated from `x` and ten from the opposing quantity `1-x`.

### Ground truth
- Original dimension: 20
- Latent dimension: 1
- Structural dimension: 2
- Generating families: two groups of 10
- Structural units: two groups of 10

### Expected behavior
MISDA should preserve one representative from each antagonistic nonlinear group while recognizing their common latent degree of freedom.


In [ ]:
case_10 = run_case("antagonistic_nonlinear_groups")


## Case 11 - Overlapping latent factors

### Purpose
Test a regular but structurally less partitionable problem, where observable families overlap in their generating factors.

### Construction
Two latent variables `(a, b)` generate objectives based on `a`, on `b`, and on the compound `a+b`, producing families of sizes 10, 4, and 6.

### Ground truth
- Original dimension: 20
- Latent dimension: 2
- Structural dimension: 2
- Generating families: 10 + 4 + 6
- Structural units: not uniquely declared

### Expected behavior
MISDA should recover two-dimensional structure without forcing the overlapping families into an artificial two-block partition.


In [ ]:
case_11 = run_case("overlapping_factors")


# Adversarial diagnostics

The final two cases deliberately exercise structures that the current pairwise graph model does not recover correctly. They are retained as documented methodological limits, not as expected successful cases. A mismatch is therefore meaningful diagnostic evidence rather than an ordinary benchmark regression.


## Case 12 - Transitive positive chain

### Purpose
Expose the transitive-chaining failure mode of pairwise positive-redundancy inference.

### Construction
Twenty independent innovations generate a cumulative triangular chain: each successive objective contains the previous cumulative signal plus a new innovation.

### Ground truth
- Original dimension: 20
- Latent dimension: 20
- Structural dimension: 20
- Generating families: one cumulative family of 20
- Structural units: 20 singletons

### Current expected limitation
The present graph construction collapses this chain to a much lower-dimensional description. The benchmark marks the resulting mismatch as the known `TRANSITIVE_CHAINING` limitation.


In [ ]:
case_12 = run_case("transitive_chain")


## Case 13 - Regime-switching dependence

### Purpose
Expose hidden dimensional structure that can be masked by regime mixing in pairwise dependence.

### Construction
Two variables `(a, b)` generate a smooth regime mixture `L(a,b)` and an additional `b` family; nonlinear transforms produce two groups of ten objectives.

### Ground truth
- Original dimension: 20
- Latent dimension: 2
- Structural dimension: 2
- Generating families: two groups of 10
- Structural units: two groups of 10

### Current expected limitation
Pairwise graph evidence currently collapses the problem to one dimension. The benchmark retains this mismatch as the known `HIDDEN_SPECTRAL_STRUCTURE` limitation.


In [ ]:
case_13 = run_case("regime_switching")


# Suite summary

The table below provides a compact cross-case view after the individual diagnostics have been inspected.


In [ ]:
diagnostic_summary = misda.compile_benchmark_summary(diagnostic_results)
diagnostic_summary
